# Disfluency restoration BERT: Colab training (self-contained)

Runtime: *Runtime > Change runtime type > T4 GPU*. No Drive access is needed: the code is embedded, Switchboard is downloaded from OpenSLR, datasets are rebuilt, and the result is downloaded as a zip. Inputs per example: a window of the disfluency-removed transcript, the G2P phones of its words and the (noisy) phone sequence of what was really said; target: the disfluencies to restore before every word.

In [ ]:
# 1) unpack the project code
import base64, io, zipfile, os
B64 = '...'
os.makedirs('/content/proj', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(B64))).extractall('/content/proj')
print(sorted(os.listdir('/content/proj')))

In [ ]:
# 2) dependencies (torch is preinstalled on Colab; pin transformers to the version used locally)
!pip -q install transformers==5.22.2 scikit-learn pyyaml
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: change the runtime type')

In [ ]:
# 3) download the free Switchboard (ISIP / Mississippi State) transcripts + lexicon from OpenSLR SLR5
import hashlib, subprocess, os
os.makedirs('/content/swb', exist_ok=True)
tar = '/content/swb/switchboard_word_alignments.tar.gz'
for base in ['https://openslr.trmal.net/resources/5', 'https://openslr.elda.org/resources/5',
             'https://openslr.magicdatatech.com/resources/5', 'https://www.openslr.org/resources/5']:
    r = subprocess.run(['curl', '-L', '--fail', '-sS', '-m', '1500', '-o', tar, base + '/switchboard_word_alignments.tar.gz'])
    if r.returncode == 0 and os.path.getsize(tar) > 40_000_000:
        print('downloaded from', base); break
else:
    raise SystemExit('could not download Switchboard from any mirror')
sha = hashlib.sha256(open(tar, 'rb').read()).hexdigest()
assert sha == '7bbc4dd5edc82b000078a028cc0a7f705e764da17cdf5027fe7888907578b106', sha
print('checksum OK', sha[:16])
!cd /content/swb && tar -xzf switchboard_word_alignments.tar.gz && ls swb_ms98_transcriptions | head -3

In [ ]:
# 4) rebuild the labelled corpus and the window datasets (same scripts as the local run)
!cd /content/proj && python -u scripts/build_processed.py --root /content/swb/swb_ms98_transcriptions --out data/processed 2>&1 | tail -25
!cd /content/proj && python -u scripts/build_seq.py --lex /content/swb/swb_ms98_transcriptions/sw-ms98-dict.text 2>&1 | tail -12

In [ ]:
# 5) start training as a DETACHED background process (fp16 autocast; 3 epochs over ~150k windows of ~230 tokens;
# expect roughly 30-60 min per epoch on a T4/L4).
# If the page loses its connection the process keeps running; cell 5b just watches its log.
import os, subprocess, time
os.makedirs('/content/ckpt/main', exist_ok=True)
LOG = '/content/ckpt/main/train.log'
env = dict(os.environ, PYTHONUNBUFFERED='1')
proc = subprocess.Popen(
    ['python', '-u', 'scripts/train_seq.py', '--epochs', '3', '--batch', '32', '--device', 'cuda', '--amp', '--save',
     '--log-every', '100', '--noise-profile', 'configs/noise_profile.json', '--out', '/content/ckpt/main'],
    cwd='/content/proj', env=env, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, start_new_session=True)
print('started pid', proc.pid, 'at', time.strftime('%H:%M:%S'))
!uptime -p

In [ ]:
# 5b) watch the log until training finishes (re-run this cell any time after a reconnect)
import subprocess, time
LOG = '/content/ckpt/main/train.log'
alive = lambda: subprocess.run(['pgrep', '-f', 'scripts/train_seq[.]py'], capture_output=True).returncode == 0
last = ''
while alive():
    cur = [l for l in open(LOG).read().splitlines() if l.strip()][-1:]
    cur = cur[0][:170] if cur else ''
    if cur != last:
        print(time.strftime('%H:%M:%S'), cur, flush=True); last = cur
    time.sleep(20)
print('training process finished')
print(''.join(open(LOG).readlines()[-18:]))

In [ ]:
# 6) shrink weights to fp16, zip and download (weights + tokenizer + metrics + log)
import torch
sd = torch.load('/content/ckpt/main/model.pt', map_location='cpu')
sd = {k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}
torch.save(sd, '/content/ckpt/main/model.pt')
!cp /content/proj/data/seq/seq_stats.json /content/ckpt/main/
!cd /content/ckpt && zip -q -r main.zip main && ls -lh main.zip
from google.colab import files
files.download('/content/ckpt/main.zip')

In [ ]:
import json
m = json.load(open('/content/ckpt/main/dev_metrics.json'))[-1]
print('slot', m['slot'], '| exact-unit', m['exact'], '| macro_f1', m['macro_f1'], '| restoration acc', m['restoration_acc_on_type_matches'])
for t, v in m['per_type'].items():
    print(f"{t:20s} P={v['p']:.3f} R={v['r']:.3f} F1={v['f1']:.3f} n={v['support']}")
print(m['word_tags'])